In [ ]:
import joblib
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_predict
)

from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)

In [ ]:
DATA_PATH = Path(
    "/content/primary_model_dataset_prepared (1).csv"
)

MODEL_PATH = Path(
    "/content/tuned_model_b_random_forest.pkl"
)

RESULTS_DIR = Path(
    "/content/counterfactual_results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Data:", DATA_PATH)
print("Model:", MODEL_PATH)
print("Results:", RESULTS_DIR)



Data: /content/primary_model_dataset_prepared (1).csv
Model: /content/tuned_model_b_random_forest.pkl
Results: /content/counterfactual_results


In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

print("\nTarget distribution:")
print(
    df["employment_outcome"]
    .value_counts()
    .sort_index()
)

print("\nMissing target:")
print(
    df["employment_outcome"]
    .isna()
    .sum()
)

print("\nDuplicate respondent IDs:")
print(
    df["respondent_id"]
    .duplicated()
    .sum()
)

Dataset shape: (191, 52)

Target distribution:
employment_outcome
0    101
1     90
Name: count, dtype: int64

Missing target:
0

Duplicate respondent IDs:
0


In [ ]:
selected_model = joblib.load(
    MODEL_PATH
)

print(
    "Loaded object type:",
    type(selected_model)
)

print(selected_model)

Loaded object type: <class 'sklearn.pipeline.Pipeline'>
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['analytical_thinking',
                                                   'resilience_flexibility_agility',
                                                   'leadership_social_influence',
                                                   'creative_thinking',
                                                   'motivation_self_awareness',
                                                   'technological_literacy',
                                   

In [ ]:
classifier = (
    selected_model
    .named_steps["classifier"]
)

print(
    "Classifier type:",
    type(classifier)
)

print("\nImportant tuned parameters:")

params_to_check = [
    "n_estimators",
    "max_depth",
    "max_features",
    "min_samples_leaf",
    "min_samples_split",
    "class_weight",
    "random_state"
]

classifier_params = (
    classifier.get_params()
)

for param in params_to_check:
    print(
        param,
        "→",
        classifier_params[param]
    )

Classifier type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>

Important tuned parameters:
n_estimators → 576
max_depth → 10
max_features → log2
min_samples_leaf → 1
min_samples_split → 2
class_weight → None
random_state → 42


In [ ]:
SKILL_FEATURES = [
    "analytical_thinking",
    "resilience_flexibility_agility",
    "leadership_social_influence",
    "creative_thinking",
    "motivation_self_awareness",
    "technological_literacy",
    "empathy_active_listening",
    "curiosity_lifelong_learning",
    "talent_management",
    "service_orientation",
]

print(
    "Number of actionable skills:",
    len(SKILL_FEATURES)
)

for feature in SKILL_FEATURES:
    print("-", feature)

Number of actionable skills: 10
- analytical_thinking
- resilience_flexibility_agility
- leadership_social_influence
- creative_thinking
- motivation_self_awareness
- technological_literacy
- empathy_active_listening
- curiosity_lifelong_learning
- talent_management
- service_orientation


In [ ]:
MODEL_B_ADDITIONAL_FEATURES = [
    "age",
    "gender",
    "marital_status",
    "household_size",

    "education_level",
    "field_of_study",
    "time_since_studies",

    "province",
    "digital_access",
    "english_communication",
    "digital_confidence",

    "formal_training",
    "training_type",
    "training_field",
    "training_duration",
    "training_relevance",
]

MODEL_B_FEATURES = (
    SKILL_FEATURES
    + MODEL_B_ADDITIONAL_FEATURES
)

print(
    "Model B feature count:",
    len(MODEL_B_FEATURES)
)

Model B feature count: 26


In [ ]:
X = (
    df[MODEL_B_FEATURES]
    .copy()
)

y = (
    df["employment_outcome"]
    .astype(int)
    .copy()
)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (191, 26)
y shape: (191,)


In [ ]:
all_indices = np.arange(
    len(df)
)

train_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train = (
    X.iloc[train_idx]
    .copy()
)

X_test = (
    X.iloc[test_idx]
    .copy()
)

y_train = (
    y.iloc[train_idx]
    .copy()
)

y_test = (
    y.iloc[test_idx]
    .copy()
)

print(
    "Train respondents:",
    len(X_train)
)

print(
    "Test respondents:",
    len(X_test)
)

print("\nTrain target:")
print(
    y_train.value_counts()
)

print("\nTest target:")
print(
    y_test.value_counts()
)

Train respondents: 152
Test respondents: 39

Train target:
employment_outcome
0    80
1    72
Name: count, dtype: int64

Test target:
employment_outcome
0    21
1    18
Name: count, dtype: int64


In [ ]:
loaded_test_prob = (
    selected_model
    .predict_proba(
        X_test
    )[:, 1]
)

print(
    "Predictions generated:",
    len(loaded_test_prob)
)

print(
    "Probability range:",
    round(
        loaded_test_prob.min(),
        4
    ),
    "to",
    round(
        loaded_test_prob.max(),
        4
    )
)

Predictions generated: 39
Probability range: 0.0005 to 0.9921


Threshold optimization

In [ ]:
threshold_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

threshold_cv_splits = list(
    threshold_cv.split(
        X_train,
        y_train
    )
)

print(
    "Threshold-selection folds:",
    len(threshold_cv_splits)
)

for fold_number, (
    fold_train_idx,
    fold_valid_idx
) in enumerate(
    threshold_cv_splits,
    start=1
):

    print(
        f"Fold {fold_number}: "
        f"train={len(fold_train_idx)}, "
        f"validation={len(fold_valid_idx)}"
    )

Threshold-selection folds: 5
Fold 1: train=121, validation=31
Fold 2: train=121, validation=31
Fold 3: train=122, validation=30
Fold 4: train=122, validation=30
Fold 5: train=122, validation=30


In [ ]:
oof_probabilities = (
    cross_val_predict(
        estimator=selected_model,
        X=X_train,
        y=y_train,
        cv=threshold_cv_splits,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]
)

print(
    "OOF predictions:",
    len(oof_probabilities)
)

print(
    "Probability min:",
    round(
        oof_probabilities.min(),
        4
    )
)

print(
    "Probability max:",
    round(
        oof_probabilities.max(),
        4
    )
)

print(
    "OOF ROC-AUC:",
    round(
        roc_auc_score(
            y_train,
            oof_probabilities
        ),
        4
    )
)

OOF predictions: 152
Probability min: 0.0007
Probability max: 0.996
OOF ROC-AUC: 0.9123


In [ ]:
candidate_thresholds = np.arange(
    0.10,
    0.901,
    0.01
)

threshold_rows = []

for threshold in candidate_thresholds:

    y_pred = (
        oof_probabilities
        >= threshold
    ).astype(int)

    bacc = (
        balanced_accuracy_score(
            y_train,
            y_pred
        )
    )

    macro_f1 = (
        f1_score(
            y_train,
            y_pred,
            average="macro",
            zero_division=0
        )
    )

    threshold_rows.append({
        "threshold":
            float(threshold),

        "balanced_accuracy":
            bacc,

        "macro_f1":
            macro_f1
    })


threshold_results = (
    pd.DataFrame(
        threshold_rows
    )
)

display(
    threshold_results
    .sort_values(
        [
            "balanced_accuracy",
            "macro_f1"
        ],
        ascending=False
    )
    .head(15)
    .round(4)
)

,threshold,balanced_accuracy,macro_f1
42,0.52,0.8347,0.8349
39,0.49,0.8292,0.8287
40,0.50,0.8292,0.8287
41,0.51,0.8292,0.8287
43,0.53,0.8271,0.8279
53,0.63,0.8264,0.8274
54,0.64,0.8264,0.8274
44,0.54,0.8201,0.8211
52,0.62,0.8201,0.8210
55,0.65,0.8194,0.8199


In [ ]:
best_threshold_row = (
    threshold_results
    .sort_values(
        [
            "balanced_accuracy",
            "macro_f1"
        ],
        ascending=[
            False,
            False
        ]
    )
    .iloc[0]
)

OPTIMIZED_THRESHOLD = float(
    best_threshold_row[
        "threshold"
    ]
)

print(
    "LOCKED OPTIMIZED THRESHOLD:",
    round(
        OPTIMIZED_THRESHOLD,
        4
    )
)

print(
    "Training OOF balanced accuracy:",
    round(
        best_threshold_row[
            "balanced_accuracy"
        ],
        4
    )
)

print(
    "Training OOF macro F1:",
    round(
        best_threshold_row[
            "macro_f1"
        ],
        4
    )
)

LOCKED OPTIMIZED THRESHOLD: 0.52
Training OOF balanced accuracy: 0.8347
Training OOF macro F1: 0.8349


In [ ]:
threshold_comparison_rows = []

for threshold_name, threshold in [
    (
        "Default",
        0.50
    ),
    (
        "Optimized",
        OPTIMIZED_THRESHOLD
    )
]:

    pred = (
        oof_probabilities
        >= threshold
    ).astype(int)

    threshold_comparison_rows.append({
        "threshold_type":
            threshold_name,

        "threshold":
            threshold,

        "balanced_accuracy":
            balanced_accuracy_score(
                y_train,
                pred
            ),

        "macro_f1":
            f1_score(
                y_train,
                pred,
                average="macro",
                zero_division=0
            )
    })


threshold_comparison = pd.DataFrame(
    threshold_comparison_rows
)

display(
    threshold_comparison.round(4)
)

,threshold_type,threshold,balanced_accuracy,macro_f1
0,Default,0.50,0.8292,0.8287
1,Optimized,0.52,0.8347,0.8349


In [ ]:
test_probabilities = (
    selected_model
    .predict_proba(
        X_test
    )[:, 1]
)

test_predictions_default = (
    test_probabilities
    >= 0.50
).astype(int)

test_predictions_optimized = (
    test_probabilities
    >= OPTIMIZED_THRESHOLD
).astype(int)

In [ ]:
holdout_threshold_rows = []

for threshold_name, predictions in [
    (
        "Default 0.50",
        test_predictions_default
    ),
    (
        "Optimized",
        test_predictions_optimized
    )
]:

    holdout_threshold_rows.append({
        "threshold_type":
            threshold_name,

        "balanced_accuracy":
            balanced_accuracy_score(
                y_test,
                predictions
            ),

        "macro_f1":
            f1_score(
                y_test,
                predictions,
                average="macro",
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                y_test,
                test_probabilities
            )
    })


holdout_threshold_comparison = (
    pd.DataFrame(
        holdout_threshold_rows
    )
)

display(
    holdout_threshold_comparison
    .round(4)
)

,threshold_type,balanced_accuracy,macro_f1,roc_auc
0,Default 0.50,0.8095,0.8126,0.9365
1,Optimized,0.8095,0.8126,0.9365


In [ ]:
print(
    "LOCKED THRESHOLD:",
    round(
        OPTIMIZED_THRESHOLD,
        4
    )
)

print()

print(
    classification_report(
        y_test,
        test_predictions_optimized,
        target_names=[
            "Active unemployed recent-search",
            "Employed"
        ],
        zero_division=0
    )
)

LOCKED THRESHOLD: 0.52

                                 precision    recall  f1-score   support

Active unemployed recent-search       0.77      0.95      0.85        21
                       Employed       0.92      0.67      0.77        18

                       accuracy                           0.82        39
                      macro avg       0.85      0.81      0.81        39
                   weighted avg       0.84      0.82      0.82        39



In [ ]:
optimized_cm = (
    confusion_matrix(
        y_test,
        test_predictions_optimized
    )
)

print(
    optimized_cm
)

print(
    "\n[[TN FP]"
    "\n [FN TP]]"
)

[[20  1]
 [ 6 12]]

[[TN FP]
 [FN TP]]


In [ ]:
threshold_results.to_csv(
    RESULTS_DIR /
    "threshold_search_results.csv",
    index=False
)

threshold_comparison.to_csv(
    RESULTS_DIR /
    "training_oof_default_vs_optimized_threshold.csv",
    index=False
)

holdout_threshold_comparison.to_csv(
    RESULTS_DIR /
    "holdout_default_vs_optimized_threshold.csv",
    index=False
)


threshold_metadata = pd.DataFrame([
    {
        "model":
            "B_RandomForest_Tuned",

        "threshold_selection_source":
            "Training-only out-of-fold predictions",

        "selection_metric":
            "Balanced accuracy",

        "tie_break_metric":
            "Macro F1",

        "optimized_threshold":
            OPTIMIZED_THRESHOLD,

        "n_splits":
            5,

        "random_state":
            RANDOM_STATE
    }
])


threshold_metadata.to_csv(
    RESULTS_DIR /
    "threshold_metadata.csv",
    index=False
)

display(
    threshold_metadata
)

,model,threshold_selection_source,selection_metric,tie_break_metric,optimized_threshold,n_splits,random_state
0,B_RandomForest_Tuned,Training-only out-of-fold predictions,Balanced accuracy,Macro F1,0.52,5,42


Counterfactual preparation

In [ ]:
IMMUTABLE_OR_FIXED_FEATURES = [
    feature
    for feature in MODEL_B_FEATURES
    if feature not in SKILL_FEATURES
]

print(
    "Actionable features:",
    len(SKILL_FEATURES)
)

print(
    "Fixed features:",
    len(IMMUTABLE_OR_FIXED_FEATURES)
)

print("\nFixed during counterfactual search:")

for feature in IMMUTABLE_OR_FIXED_FEATURES:
    print("-", feature)

Actionable features: 10
Fixed features: 16

Fixed during counterfactual search:
- age
- gender
- marital_status
- household_size
- education_level
- field_of_study
- time_since_studies
- province
- digital_access
- english_communication
- digital_confidence
- formal_training
- training_type
- training_field
- training_duration
- training_relevance


In [ ]:
skill_validation = pd.DataFrame({
    "min":
        df[SKILL_FEATURES]
        .min(),

    "max":
        df[SKILL_FEATURES]
        .max(),

    "unique_values":
        [
            sorted(
                df[col]
                .dropna()
                .unique()
                .tolist()
            )
            for col
            in SKILL_FEATURES
        ]
})

display(
    skill_validation
)

,min,max,unique_values
analytical_thinking,1.0,5.0,"[1.0, 2.0, 3.0, 4.0, 5.0]"
resilience_flexibility_agility,1.0,5.0,"[1.0, 2.0, 3.0, 4.0, 5.0]"
leadership_social_influence,1.0,5.0,"[1.0, 2.0, 3.0, 4.0, 5.0]"
creative_thinking,1.0,5.0,"[1.0, 2.0, 3.0, 4.0, 5.0]"
motivation_self_awareness,1.0,5.0,"[1.0, 2.0, 3.0, 4.0, 5.0]"
technological_literacy,1.0,5.0,"[1.0, 2.0, 3.0, 4.0, 5.0]"
empathy_active_listening,1.0,5.0,"[1.0, 2.0, 3.0, 4.0, 5.0]"
curiosity_lifelong_learning,1.0,5.0,"[1.0, 2.0, 3.0, 4.0, 5.0]"
talent_management,1.0,5.0,"[1.0, 2.0, 3.0, 4.0, 5.0]"
service_orientation,1.0,5.0,"[1.0, 2.0, 3.0, 4.0, 5.0]"


In [ ]:
SKILL_MIN = 1
SKILL_MAX = 5
SKILL_STEP = 1

print(
    "Skill search range:",
    SKILL_MIN,
    "to",
    SKILL_MAX
)

print(
    "Step size:",
    SKILL_STEP
)

Skill search range: 1 to 5
Step size: 1


In [ ]:
def predict_employment_probability(
    row_df
):

    return float(
        selected_model
        .predict_proba(
            row_df[
                MODEL_B_FEATURES
            ]
        )[0, 1]
    )


def classify_with_locked_threshold(
    probability
):

    return int(
        probability
        >= OPTIMIZED_THRESHOLD
    )

In [ ]:
example_index = (
    X_test.index[0]
)

example_row = (
    X.loc[
        [example_index]
    ]
    .copy()
)

example_probability = (
    predict_employment_probability(
        example_row
    )
)

example_class = (
    classify_with_locked_threshold(
        example_probability
    )
)

print(
    "Respondent index:",
    example_index
)

print(
    "Probability:",
    round(
        example_probability,
        4
    )
)

print(
    "Locked threshold:",
    round(
        OPTIMIZED_THRESHOLD,
        4
    )
)

print(
    "Predicted class:",
    example_class
)

display(
    example_row[
        SKILL_FEATURES
    ]
)

Respondent index: 153
Probability: 0.4638
Locked threshold: 0.52
Predicted class: 0


,analytical_thinking,resilience_flexibility_agility,leadership_social_influence,creative_thinking,motivation_self_awareness,technological_literacy,empathy_active_listening,curiosity_lifelong_learning,talent_management,service_orientation
153,3.0,3.0,5.0,5.0,4.0,1.0,5.0,5.0,5.0,4.0


In [ ]:
def evaluate_counterfactual(
    original_row,
    counterfactual_row
):
    original_probability = (
        predict_employment_probability(
            original_row
        )
    )

    counterfactual_probability = (
        predict_employment_probability(
            counterfactual_row
        )
    )

    return {
        "original_probability":
            original_probability,

        "counterfactual_probability":
            counterfactual_probability,

        "probability_gain":
            (
                counterfactual_probability
                - original_probability
            ),

        "original_class":
            classify_with_locked_threshold(
                original_probability
            ),

        "counterfactual_class":
            classify_with_locked_threshold(
                counterfactual_probability
            ),

        "valid_flip":
            (
                original_probability
                < OPTIMIZED_THRESHOLD
                and
                counterfactual_probability
                >= OPTIMIZED_THRESHOLD
            )
    }

In [ ]:
def validate_counterfactual_constraints(
    original_row,
    counterfactual_row
):
    # All non-skill Model B variables
    # MUST remain exactly unchanged.
    fixed_unchanged = all(
        original_row.iloc[0][feature]
        ==
        counterfactual_row.iloc[0][feature]

        for feature
        in IMMUTABLE_OR_FIXED_FEATURES
    )

    # Skills may only stay the same
    # or increase.
    skills_not_decreased = all(
        counterfactual_row.iloc[0][skill]
        >=
        original_row.iloc[0][skill]

        for skill
        in SKILL_FEATURES
    )

    # Skills must remain within
    # the original 1-5 scale.
    skills_within_bounds = all(
        SKILL_MIN
        <=
        counterfactual_row.iloc[0][skill]
        <=
        SKILL_MAX

        for skill
        in SKILL_FEATURES
    )

    return (
        fixed_unchanged
        and
        skills_not_decreased
        and
        skills_within_bounds
    )

In [ ]:
def search_single_skill_counterfactuals(
    original_row
):
    original_probability = (
        predict_employment_probability(
            original_row
        )
    )

    results = []

    for skill in SKILL_FEATURES:

        current_value = int(
            original_row.iloc[0][skill]
        )

        # Cannot improve beyond 5.
        if current_value >= SKILL_MAX:
            continue

        for new_value in range(
            current_value + SKILL_STEP,
            SKILL_MAX + 1,
            SKILL_STEP
        ):

            cf_row = (
                original_row.copy()
            )

            cf_row.loc[
                cf_row.index[0],
                skill
            ] = new_value

            if not validate_counterfactual_constraints(
                original_row,
                cf_row
            ):
                continue

            cf_probability = (
                predict_employment_probability(
                    cf_row
                )
            )

            results.append({
                "skill":
                    skill,

                "original_value":
                    current_value,

                "counterfactual_value":
                    new_value,

                "change":
                    new_value
                    - current_value,

                "original_probability":
                    original_probability,

                "counterfactual_probability":
                    cf_probability,

                "probability_gain":
                    cf_probability
                    - original_probability,

                "crosses_threshold":
                    cf_probability
                    >= OPTIMIZED_THRESHOLD
            })

    return (
        pd.DataFrame(results)
    )

In [ ]:
single_skill_results = (
    search_single_skill_counterfactuals(
        example_row
    )
)

single_skill_results = (
    single_skill_results
    .sort_values(
        [
            "crosses_threshold",
            "probability_gain"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

display(
    single_skill_results.round(4)
)

,skill,original_value,counterfactual_value,change,original_probability,counterfactual_probability,probability_gain,crosses_threshold
0,technological_literacy,1,5,4,0.4638,0.4941,0.0303,False
1,technological_literacy,1,4,3,0.4638,0.4794,0.0156,False
2,resilience_flexibility_agility,3,4,1,0.4638,0.4665,0.0027,False
3,analytical_thinking,3,4,1,0.4638,0.4653,0.0015,False
4,technological_literacy,1,3,2,0.4638,0.4599,-0.0039,False
5,service_orientation,4,5,1,0.4638,0.4592,-0.0046,False
6,motivation_self_awareness,4,5,1,0.4638,0.4553,-0.0085,False
7,technological_literacy,1,2,1,0.4638,0.4526,-0.0113,False
8,resilience_flexibility_agility,3,5,2,0.4638,0.4496,-0.0142,False
9,analytical_thinking,3,5,2,0.4638,0.4384,-0.0254,False


In [ ]:
successful_single_skill = (
    single_skill_results[
        single_skill_results[
            "crosses_threshold"
        ]
    ]
    .copy()
)

print(
    "Successful single-skill "
    "counterfactuals:",
    len(successful_single_skill)
)

if len(
    successful_single_skill
) > 0:

    display(
        successful_single_skill[
            [
                "skill",
                "original_value",
                "counterfactual_value",
                "change",
                "original_probability",
                "counterfactual_probability",
                "probability_gain"
            ]
        ].round(4)
    )

else:

    print(
        "No single skill improvement "
        "crossed the locked threshold."
    )

Successful single-skill counterfactuals: 0
No single skill improvement crossed the locked threshold.


In [ ]:
def get_feasible_skill_values(
    original_row
):
    feasible_values = {}

    for skill in SKILL_FEATURES:

        current_value = int(
            original_row.iloc[0][skill]
        )

        feasible_values[skill] = list(
            range(
                current_value,
                SKILL_MAX + 1,
                SKILL_STEP
            )
        )

    return feasible_values

In [ ]:
example_feasible_values = (
    get_feasible_skill_values(
        example_row
    )
)

total_combinations = 1

for skill, values in (
    example_feasible_values.items()
):

    print(
        skill,
        "→",
        values
    )

    total_combinations *= len(values)


print(
    "\nTotal feasible combinations:",
    total_combinations
)

analytical_thinking → [3, 4, 5]
resilience_flexibility_agility → [3, 4, 5]
leadership_social_influence → [5]
creative_thinking → [5]
motivation_self_awareness → [4, 5]
technological_literacy → [1, 2, 3, 4, 5]
empathy_active_listening → [5]
curiosity_lifelong_learning → [5]
talent_management → [5]
service_orientation → [4, 5]

Total feasible combinations: 180


In [ ]:
from itertools import product

In [ ]:
def search_skill_counterfactuals(
    original_row,
    max_combinations=200000
):
    original_probability = (
        predict_employment_probability(
            original_row
        )
    )

    feasible_values = (
        get_feasible_skill_values(
            original_row
        )
    )

    value_lists = [
        feasible_values[skill]
        for skill in SKILL_FEATURES
    ]

    total_combinations = int(
        np.prod(
            [
                len(values)
                for values in value_lists
            ]
        )
    )

    print(
        "Feasible combinations:",
        total_combinations
    )

    if total_combinations > max_combinations:

        raise ValueError(
            "Search space exceeds "
            f"{max_combinations:,} combinations."
        )

    successful = []

    for combination in product(
        *value_lists
    ):

        # Skip original profile.
        original_values = tuple(
            int(
                original_row.iloc[0][skill]
            )
            for skill in SKILL_FEATURES
        )

        if combination == original_values:
            continue

        cf_row = (
            original_row.copy()
        )

        changed_skills = []
        total_change = 0

        for skill, new_value in zip(
            SKILL_FEATURES,
            combination
        ):

            old_value = int(
                original_row.iloc[0][skill]
            )

            cf_row.loc[
                cf_row.index[0],
                skill
            ] = new_value

            if new_value > old_value:

                changed_skills.append(
                    {
                        "skill": skill,
                        "from": old_value,
                        "to": new_value,
                        "increase":
                            new_value - old_value
                    }
                )

                total_change += (
                    new_value - old_value
                )

        if not validate_counterfactual_constraints(
            original_row,
            cf_row
        ):
            continue

        cf_probability = (
            predict_employment_probability(
                cf_row
            )
        )

        if (
            cf_probability
            >= OPTIMIZED_THRESHOLD
        ):

            successful.append({
                "counterfactual_probability":
                    cf_probability,

                "probability_gain":
                    cf_probability
                    - original_probability,

                "number_of_skills_changed":
                    len(changed_skills),

                "total_skill_increase":
                    total_change,

                "changed_skills":
                    changed_skills
            })

    return successful

In [ ]:
example_counterfactuals = (
    search_skill_counterfactuals(
        example_row
    )
)

print(
    "\nSuccessful counterfactuals:",
    len(example_counterfactuals)
)

Feasible combinations: 180

Successful counterfactuals: 0


In [ ]:
example_counterfactuals_sorted = sorted(
    example_counterfactuals,
    key=lambda x: (
        x[
            "number_of_skills_changed"
        ],
        x[
            "total_skill_increase"
        ],
        -x[
            "counterfactual_probability"
        ]
    )
)

print(
    "Number of valid solutions:",
    len(
        example_counterfactuals_sorted
    )
)

Number of valid solutions: 0


In [ ]:
top_cf_rows = []

for rank, cf in enumerate(
    example_counterfactuals_sorted[:10],
    start=1
):

    changes_text = "; ".join(
        [
            (
                f"{change['skill']}: "
                f"{change['from']} → "
                f"{change['to']}"
            )
            for change
            in cf["changed_skills"]
        ]
    )

    top_cf_rows.append({
        "rank":
            rank,

        "skills_changed":
            cf[
                "number_of_skills_changed"
            ],

        "total_skill_increase":
            cf[
                "total_skill_increase"
            ],

        "counterfactual_probability":
            cf[
                "counterfactual_probability"
            ],

        "probability_gain":
            cf[
                "probability_gain"
            ],

        "changes":
            changes_text
    })


top_counterfactuals_df = (
    pd.DataFrame(
        top_cf_rows
    )
)

display(
    top_counterfactuals_df
    .round(4)
)

""


In [ ]:
all_probabilities = (
    selected_model
    .predict_proba(
        X[MODEL_B_FEATURES]
    )[:, 1]
)

audit_df = df.copy()

audit_df["predicted_probability"] = (
    all_probabilities
)

audit_df["predicted_class"] = (
    audit_df["predicted_probability"]
    >= OPTIMIZED_THRESHOLD
).astype(int)

below_threshold_df = (
    audit_df[
        audit_df["predicted_class"] == 0
    ]
    .copy()
)

print(
    "Total respondents:",
    len(audit_df)
)

print(
    "Predicted below threshold:",
    len(below_threshold_df)
)

print(
    "Predicted at/above threshold:",
    (
        audit_df["predicted_class"] == 1
    ).sum()
)

print("\nObserved outcomes among below-threshold respondents:")

print(
    below_threshold_df[
        "employment_outcome"
    ].value_counts()
)

Total respondents: 191
Predicted below threshold: 106
Predicted at/above threshold: 85

Observed outcomes among below-threshold respondents:
employment_outcome
0    100
1      6
Name: count, dtype: int64


Monotonicity audit

In [ ]:
monotonicity_rows = []

for respondent_index in X.index:

    original_row = (
        X.loc[
            [respondent_index]
        ]
        .copy()
    )

    original_probability = (
        predict_employment_probability(
            original_row
        )
    )

    for skill in SKILL_FEATURES:

        current_value = int(
            original_row.iloc[0][skill]
        )

        # Cannot increase a skill already at 5.
        if current_value >= SKILL_MAX:
            continue

        cf_row = original_row.copy()

        cf_row.loc[
            respondent_index,
            skill
        ] = current_value + 1

        cf_probability = (
            predict_employment_probability(
                cf_row
            )
        )

        probability_change = (
            cf_probability
            - original_probability
        )

        if probability_change > 1e-10:
            direction = "increase"

        elif probability_change < -1e-10:
            direction = "decrease"

        else:
            direction = "no_change"

        monotonicity_rows.append({
            "respondent_index":
                respondent_index,

            "skill":
                skill,

            "original_value":
                current_value,

            "new_value":
                current_value + 1,

            "original_probability":
                original_probability,

            "counterfactual_probability":
                cf_probability,

            "probability_change":
                probability_change,

            "direction":
                direction
        })


monotonicity_df = pd.DataFrame(
    monotonicity_rows
)

print(
    "Total one-level interventions:",
    len(monotonicity_df)
)

display(
    monotonicity_df.head()
)

Total one-level interventions: 1176


,respondent_index,skill,original_value,new_value,original_probability,counterfactual_probability,probability_change,direction
0,0,technological_literacy,3,4,0.832479,0.797886,-0.034593,decrease
1,1,resilience_flexibility_agility,4,5,0.000548,0.019645,0.019097,increase
2,1,technological_literacy,4,5,0.000548,0.016398,0.015850,increase
3,2,analytical_thinking,4,5,0.253110,0.255398,0.002288,increase
4,2,creative_thinking,4,5,0.253110,0.258314,0.005204,increase


In [ ]:
overall_monotonicity = (
    monotonicity_df[
        "direction"
    ]
    .value_counts()
    .rename_axis("direction")
    .reset_index(name="count")
)

overall_monotonicity[
    "percentage"
] = (
    overall_monotonicity["count"]
    /
    overall_monotonicity["count"].sum()
    * 100
)

display(
    overall_monotonicity.round(2)
)

,direction,count,percentage
0,decrease,602,51.19
1,increase,571,48.55
2,no_change,3,0.26


In [ ]:
skill_monotonicity = (
    monotonicity_df
    .groupby(
        [
            "skill",
            "direction"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

for column in [
    "increase",
    "decrease",
    "no_change"
]:

    if column not in skill_monotonicity.columns:
        skill_monotonicity[column] = 0


skill_monotonicity[
    "total"
] = (
    skill_monotonicity[
        [
            "increase",
            "decrease",
            "no_change"
        ]
    ]
    .sum(axis=1)
)


skill_monotonicity[
    "increase_pct"
] = (
    skill_monotonicity["increase"]
    /
    skill_monotonicity["total"]
    * 100
)


skill_monotonicity[
    "decrease_pct"
] = (
    skill_monotonicity["decrease"]
    /
    skill_monotonicity["total"]
    * 100
)


skill_monotonicity[
    "no_change_pct"
] = (
    skill_monotonicity["no_change"]
    /
    skill_monotonicity["total"]
    * 100
)


skill_monotonicity = (
    skill_monotonicity
    .sort_values(
        "increase_pct",
        ascending=False
    )
)

display(
    skill_monotonicity[
        [
            "increase",
            "decrease",
            "no_change",
            "increase_pct",
            "decrease_pct",
            "no_change_pct"
        ]
    ].round(2)
)

direction,increase,decrease,no_change,increase_pct,decrease_pct,no_change_pct
skill,,,,,,
technological_literacy,93,45,0,67.39,32.61,0.00
resilience_flexibility_agility,80,65,0,55.17,44.83,0.00
creative_thinking,67,58,0,53.60,46.40,0.00
analytical_thinking,66,69,0,48.89,51.11,0.00
motivation_self_awareness,62,68,3,46.62,51.13,2.26
curiosity_lifelong_learning,40,50,0,44.44,55.56,0.00
service_orientation,49,62,0,44.14,55.86,0.00
talent_management,46,61,0,42.99,57.01,0.00
empathy_active_listening,34,61,0,35.79,64.21,0.00


In [ ]:
skill_effect_summary = (
    monotonicity_df
    .groupby("skill")
    .agg(
        n_interventions=(
            "probability_change",
            "size"
        ),

        mean_probability_change=(
            "probability_change",
            "mean"
        ),

        median_probability_change=(
            "probability_change",
            "median"
        ),

        min_probability_change=(
            "probability_change",
            "min"
        ),

        max_probability_change=(
            "probability_change",
            "max"
        )
    )
    .sort_values(
        "mean_probability_change",
        ascending=False
    )
)

display(
    skill_effect_summary.round(4)
)

,n_interventions,mean_probability_change,median_probability_change,min_probability_change,max_probability_change
skill,,,,,
technological_literacy,138,0.0138,0.0150,-0.0346,0.0695
resilience_flexibility_agility,145,0.0000,0.0052,-0.0535,0.0426
analytical_thinking,135,-0.0002,-0.0007,-0.0637,0.0407
motivation_self_awareness,133,-0.0031,-0.0014,-0.0662,0.0686
talent_management,107,-0.0038,-0.0057,-0.0547,0.0779
creative_thinking,125,-0.0039,0.0040,-0.0577,0.0648
curiosity_lifelong_learning,90,-0.0074,-0.0048,-0.0501,0.0491
empathy_active_listening,95,-0.0086,-0.0096,-0.0522,0.0309
service_orientation,111,-0.0091,-0.0088,-0.0937,0.0449


In [ ]:
single_cf_audit_rows = []

for respondent_index in (
    below_threshold_df.index
):

    original_row = (
        X.loc[
            [respondent_index]
        ]
        .copy()
    )

    original_probability = (
        predict_employment_probability(
            original_row
        )
    )

    results = (
        search_single_skill_counterfactuals(
            original_row
        )
    )

    if len(results) > 0:

        successful = (
            results[
                results[
                    "crosses_threshold"
                ]
            ]
            .copy()
        )

    else:

        successful = pd.DataFrame()

    has_single_cf = (
        len(successful) > 0
    )

    if has_single_cf:

        successful = (
            successful
            .sort_values(
                [
                    "change",
                    "counterfactual_probability"
                ],
                ascending=[
                    True,
                    False
                ]
            )
        )

        best = successful.iloc[0]

        best_skill = best["skill"]
        best_from = best["original_value"]
        best_to = best["counterfactual_value"]
        best_probability = (
            best[
                "counterfactual_probability"
            ]
        )

    else:

        best_skill = None
        best_from = None
        best_to = None
        best_probability = None

    single_cf_audit_rows.append({
        "respondent_index":
            respondent_index,

        "original_probability":
            original_probability,

        "has_single_skill_cf":
            has_single_cf,

        "best_skill":
            best_skill,

        "from_value":
            best_from,

        "to_value":
            best_to,

        "counterfactual_probability":
            best_probability
    })


single_cf_audit = pd.DataFrame(
    single_cf_audit_rows
)

display(
    single_cf_audit.head()
)

,respondent_index,original_probability,has_single_skill_cf,best_skill,from_value,to_value,counterfactual_probability
0,1,0.000548,False,None,NaN,NaN,NaN
1,2,0.253110,False,None,NaN,NaN,NaN
2,4,0.412850,False,None,NaN,NaN,NaN
3,5,0.153557,False,None,NaN,NaN,NaN
4,7,0.292955,False,None,NaN,NaN,NaN


In [ ]:
n_below = len(
    single_cf_audit
)

n_single_success = int(
    single_cf_audit[
        "has_single_skill_cf"
    ].sum()
)

single_success_rate = (
    n_single_success
    /
    n_below
    * 100
)

print(
    "Below-threshold respondents:",
    n_below
)

print(
    "With ≥1 single-skill counterfactual:",
    n_single_success
)

print(
    "Without single-skill counterfactual:",
    n_below - n_single_success
)

print(
    "Single-skill CF coverage:",
    round(
        single_success_rate,
        2
    ),
    "%"
)

Below-threshold respondents: 106
With ≥1 single-skill counterfactual: 1
Without single-skill counterfactual: 105
Single-skill CF coverage: 0.94 %


In [ ]:
successful_skill_counts = (
    single_cf_audit[
        single_cf_audit[
            "has_single_skill_cf"
        ]
    ][
        "best_skill"
    ]
    .value_counts()
    .rename_axis("skill")
    .reset_index(
        name="respondents"
    )
)

if len(
    successful_skill_counts
) > 0:

    successful_skill_counts[
        "percentage_of_single_cf_cases"
    ] = (
        successful_skill_counts[
            "respondents"
        ]
        /
        n_single_success
        * 100
    )

display(
    successful_skill_counts
)

,skill,respondents,percentage_of_single_cf_cases
0,technological_literacy,1,100.0


In [ ]:
search_space_rows = []

for respondent_index in (
    below_threshold_df.index
):

    row = (
        X.loc[
            [respondent_index]
        ]
    )

    feasible_values = (
        get_feasible_skill_values(
            row
        )
    )

    search_size = int(
        np.prod(
            [
                len(values)
                for values
                in feasible_values.values()
            ]
        )
    )

    search_space_rows.append({
        "respondent_index":
            respondent_index,

        "predicted_probability":
            float(
                below_threshold_df.loc[
                    respondent_index,
                    "predicted_probability"
                ]
            ),

        "search_space_size":
            search_size
    })


search_space_df = pd.DataFrame(
    search_space_rows
)

display(
    search_space_df[
        "search_space_size"
    ].describe()
)

print(
    "\nLargest search space:",
    search_space_df[
        "search_space_size"
    ].max()
)

,search_space_size
count,1.060000e+02
mean,3.516015e+05
std,1.183342e+06
min,1.000000e+00
25%,1.200000e+01
50%,8.400000e+01
75%,5.184000e+03
max,6.250000e+06



Largest search space: 6250000


In [ ]:
display(
    search_space_df
    .sort_values(
        "search_space_size",
        ascending=False
    )
    .head(20)
)

,respondent_index,predicted_probability,search_space_size
99,179,0.078057,6250000
97,174,0.078057,6250000
87,155,0.078057,6250000
12,17,0.356753,5000000
21,27,0.120732,1536000
77,138,0.120732,1536000
54,92,0.120732,1536000
16,22,0.120732,1536000
86,154,0.120732,1536000
13,18,0.120732,1536000


In [ ]:
from itertools import (
    combinations,
    product
)

In [ ]:
def search_minimal_skill_counterfactuals(
    original_row,
    max_changed_skills=10
):

    original_probability = (
        predict_employment_probability(
            original_row
        )
    )

    # Already positive → no recourse required.
    if (
        original_probability
        >= OPTIMIZED_THRESHOLD
    ):
        return []

    solutions = []

    for n_changed in range(
        1,
        max_changed_skills + 1
    ):

        level_solutions = []

        changeable_skills = [
            skill
            for skill in SKILL_FEATURES
            if int(
                original_row.iloc[0][skill]
            ) < SKILL_MAX
        ]

        if (
            len(changeable_skills)
            < n_changed
        ):
            break

        for selected_skills in combinations(
            changeable_skills,
            n_changed
        ):

            possible_new_values = []

            for skill in selected_skills:

                current_value = int(
                    original_row.iloc[0][skill]
                )

                possible_new_values.append(
                    list(
                        range(
                            current_value + 1,
                            SKILL_MAX + 1
                        )
                    )
                )

            for new_values in product(
                *possible_new_values
            ):

                cf_row = (
                    original_row.copy()
                )

                changes = []
                total_increase = 0

                for skill, new_value in zip(
                    selected_skills,
                    new_values
                ):

                    old_value = int(
                        original_row.iloc[0][skill]
                    )

                    cf_row.loc[
                        cf_row.index[0],
                        skill
                    ] = new_value

                    increase = (
                        new_value
                        - old_value
                    )

                    total_increase += (
                        increase
                    )

                    changes.append({
                        "skill":
                            skill,

                        "from":
                            old_value,

                        "to":
                            new_value,

                        "increase":
                            increase
                    })

                if not validate_counterfactual_constraints(
                    original_row,
                    cf_row
                ):
                    continue

                cf_probability = (
                    predict_employment_probability(
                        cf_row
                    )
                )

                if (
                    cf_probability
                    >= OPTIMIZED_THRESHOLD
                ):

                    level_solutions.append({
                        "original_probability":
                            original_probability,

                        "counterfactual_probability":
                            cf_probability,

                        "probability_gain":
                            (
                                cf_probability
                                - original_probability
                            ),

                        "number_of_skills_changed":
                            n_changed,

                        "total_skill_increase":
                            total_increase,

                        "changed_skills":
                            changes
                    })

        # Minimality:
        # once solutions exist for n skills,
        # do not search n+1 skills.
        if len(level_solutions) > 0:

            level_solutions = sorted(
                level_solutions,
                key=lambda x: (
                    x[
                        "total_skill_increase"
                    ],
                    -x[
                        "counterfactual_probability"
                    ]
                )
            )

            return level_solutions

    return []

In [ ]:
minimal_cf_153 = (
    search_minimal_skill_counterfactuals(
        example_row
    )
)

print(
    "Minimal counterfactuals found:",
    len(minimal_cf_153)
)

if len(minimal_cf_153) > 0:

    print(
        "Minimum number of skills changed:",
        minimal_cf_153[0][
            "number_of_skills_changed"
        ]
    )

    print(
        "Minimum total skill increase:",
        minimal_cf_153[0][
            "total_skill_increase"
        ]
    )

    print(
        "Probability:",
        round(
            minimal_cf_153[0][
                "counterfactual_probability"
            ],
            4
        )
    )

    print(
        "\nChanges:"
    )

    for change in (
        minimal_cf_153[0][
            "changed_skills"
        ]
    ):

        print(
            change["skill"],
            ":",
            change["from"],
            "→",
            change["to"]
        )

else:

    print(
        "No feasible skill-only "
        "counterfactual exists."
    )

Minimal counterfactuals found: 0
No feasible skill-only counterfactual exists.


In [ ]:
negative_indices = set(
    below_threshold_df.index
)

below_threshold_monotonicity = (
    monotonicity_df[
        monotonicity_df[
            "respondent_index"
        ].isin(negative_indices)
    ]
    .copy()
)

print(
    "Interventions among below-threshold respondents:",
    len(below_threshold_monotonicity)
)

below_overall = (
    below_threshold_monotonicity[
        "direction"
    ]
    .value_counts()
    .rename_axis("direction")
    .reset_index(name="count")
)

below_overall["percentage"] = (
    below_overall["count"]
    / below_overall["count"].sum()
    * 100
)

display(
    below_overall.round(2)
)

Interventions among below-threshold respondents: 620


,direction,count,percentage
0,increase,528,85.16
1,decrease,89,14.35
2,no_change,3,0.48


In [ ]:
below_skill_monotonicity = (
    below_threshold_monotonicity
    .groupby(
        ["skill", "direction"]
    )
    .size()
    .unstack(fill_value=0)
)

for col in [
    "increase",
    "decrease",
    "no_change"
]:
    if col not in below_skill_monotonicity.columns:
        below_skill_monotonicity[col] = 0


below_skill_monotonicity["total"] = (
    below_skill_monotonicity[
        [
            "increase",
            "decrease",
            "no_change"
        ]
    ].sum(axis=1)
)

below_skill_monotonicity["increase_pct"] = (
    below_skill_monotonicity["increase"]
    / below_skill_monotonicity["total"]
    * 100
)

below_skill_monotonicity["decrease_pct"] = (
    below_skill_monotonicity["decrease"]
    / below_skill_monotonicity["total"]
    * 100
)

below_skill_monotonicity["no_change_pct"] = (
    below_skill_monotonicity["no_change"]
    / below_skill_monotonicity["total"]
    * 100
)

below_skill_monotonicity = (
    below_skill_monotonicity
    .sort_values(
        "increase_pct",
        ascending=False
    )
)

display(
    below_skill_monotonicity[
        [
            "increase",
            "decrease",
            "no_change",
            "increase_pct",
            "decrease_pct",
            "no_change_pct"
        ]
    ].round(2)
)

direction,increase,decrease,no_change,increase_pct,decrease_pct,no_change_pct
skill,,,,,,
technological_literacy,78,4,0,95.12,4.88,0.00
resilience_flexibility_agility,76,5,0,93.83,6.17,0.00
creative_thinking,60,7,0,89.55,10.45,0.00
analytical_thinking,64,8,0,88.89,11.11,0.00
service_orientation,49,7,0,87.50,12.50,0.00
motivation_self_awareness,58,9,3,82.86,12.86,4.29
curiosity_lifelong_learning,37,8,0,82.22,17.78,0.00
leadership_social_influence,33,12,0,73.33,26.67,0.00
talent_management,40,15,0,72.73,27.27,0.00


In [ ]:
below_skill_effects = (
    below_threshold_monotonicity
    .groupby("skill")
    .agg(
        n_interventions=(
            "probability_change",
            "size"
        ),

        mean_change=(
            "probability_change",
            "mean"
        ),

        median_change=(
            "probability_change",
            "median"
        ),

        std_change=(
            "probability_change",
            "std"
        ),

        min_change=(
            "probability_change",
            "min"
        ),

        max_change=(
            "probability_change",
            "max"
        ),

        positive_rate=(
            "probability_change",
            lambda x: (
                x > 1e-10
            ).mean() * 100
        )
    )
    .sort_values(
        "mean_change",
        ascending=False
    )
)

display(
    below_skill_effects.round(4)
)

,n_interventions,mean_change,median_change,std_change,min_change,max_change,positive_rate
skill,,,,,,,
technological_literacy,82,0.0256,0.0201,0.0169,-0.0113,0.0695,95.1220
resilience_flexibility_agility,81,0.0172,0.0186,0.0153,-0.0339,0.0426,93.8272
creative_thinking,67,0.0164,0.0160,0.0180,-0.0211,0.0648,89.5522
motivation_self_awareness,70,0.0159,0.0136,0.0171,-0.0146,0.0686,82.8571
talent_management,55,0.0156,0.0156,0.0207,-0.0214,0.0779,72.7273
curiosity_lifelong_learning,45,0.0144,0.0139,0.0177,-0.0376,0.0491,82.2222
analytical_thinking,72,0.0135,0.0125,0.0129,-0.0213,0.0407,88.8889
service_orientation,56,0.0130,0.0139,0.0151,-0.0289,0.0449,87.5000
leadership_social_influence,45,0.0129,0.0148,0.0358,-0.0743,0.0640,73.3333


In [ ]:
below_threshold_distance = (
    below_threshold_df[
        [
            "predicted_probability"
        ]
    ]
    .copy()
)

below_threshold_distance[
    "distance_to_threshold"
] = (
    OPTIMIZED_THRESHOLD
    - below_threshold_distance[
        "predicted_probability"
    ]
)

display(
    below_threshold_distance[
        "distance_to_threshold"
    ].describe()
)

print("\nClosest 10 negative predictions:")

display(
    below_threshold_distance
    .sort_values(
        "distance_to_threshold"
    )
    .head(10)
    .round(4)
)

,distance_to_threshold
count,106.000000
mean,0.358284
std,0.123078
min,0.033879
25%,0.283124
50%,0.381589
75%,0.453638
max,0.519452



Closest 10 negative predictions:


,predicted_probability,distance_to_threshold
106,0.4861,0.0339
12,0.4696,0.0504
153,0.4638,0.0562
156,0.4384,0.0816
29,0.4323,0.0877
130,0.4323,0.0877
107,0.4281,0.0919
34,0.4266,0.0934
4,0.4129,0.1071
17,0.3568,0.1632


In [ ]:
distance_bands = pd.cut(
    below_threshold_df[
        "predicted_probability"
    ],
    bins=[
        0.0,
        0.20,
        0.30,
        0.40,
        0.45,
        0.50,
        OPTIMIZED_THRESHOLD
    ],
    include_lowest=True
)

distance_summary = (
    distance_bands
    .value_counts()
    .sort_index()
    .rename_axis(
        "probability_range"
    )
    .reset_index(
        name="respondents"
    )
)

distance_summary[
    "percentage"
] = (
    distance_summary[
        "respondents"
    ]
    / len(
        below_threshold_df
    )
    * 100
)

display(
    distance_summary.round(2)
)

,probability_range,respondents,percentage
0,"(-0.001, 0.2]",70,66.04
1,"(0.2, 0.3]",24,22.64
2,"(0.3, 0.4]",3,2.83
3,"(0.4, 0.45]",6,5.66
4,"(0.45, 0.5]",3,2.83
5,"(0.5, 0.52]",0,0.00


In [ ]:
best_single_rows = []

for respondent_index in below_threshold_df.index:

    original_row = (
        X.loc[
            [respondent_index]
        ]
        .copy()
    )

    original_probability = (
        predict_employment_probability(
            original_row
        )
    )

    results = (
        search_single_skill_counterfactuals(
            original_row
        )
    )

    if len(results) > 0:

        best_idx = (
            results[
                "counterfactual_probability"
            ].idxmax()
        )

        best = results.loc[
            best_idx
        ]

        best_single_rows.append({
            "respondent_index":
                respondent_index,

            "original_probability":
                original_probability,

            "best_probability":
                best[
                    "counterfactual_probability"
                ],

            "maximum_gain":
                best[
                    "counterfactual_probability"
                ]
                - original_probability,

            "best_skill":
                best["skill"],

            "original_value":
                best[
                    "original_value"
                ],

            "recommended_value":
                best[
                    "counterfactual_value"
                ],

            "crosses_threshold":
                best[
                    "counterfactual_probability"
                ]
                >= OPTIMIZED_THRESHOLD
        })

    else:

        best_single_rows.append({
            "respondent_index":
                respondent_index,

            "original_probability":
                original_probability,

            "best_probability":
                original_probability,

            "maximum_gain":
                0.0,

            "best_skill":
                None,

            "original_value":
                None,

            "recommended_value":
                None,

            "crosses_threshold":
                False
        })


best_single_df = pd.DataFrame(
    best_single_rows
)

display(
    best_single_df
    .sort_values(
        "maximum_gain",
        ascending=False
    )
    .head(20)
    .round(4)
)

,respondent_index,original_probability,best_probability,maximum_gain,best_skill,original_value,recommended_value,crosses_threshold
72,125,0.1384,0.2529,0.1145,resilience_flexibility_agility,1.0,5.0,False
45,72,0.1384,0.2529,0.1145,resilience_flexibility_agility,1.0,5.0,False
80,145,0.1384,0.2529,0.1145,resilience_flexibility_agility,1.0,5.0,False
48,78,0.2351,0.3462,0.1110,technological_literacy,2.0,5.0,False
100,180,0.0288,0.1345,0.1057,resilience_flexibility_agility,3.0,5.0,False
67,115,0.0288,0.1345,0.1057,resilience_flexibility_agility,3.0,5.0,False
58,101,0.0288,0.1345,0.1057,resilience_flexibility_agility,3.0,5.0,False
105,189,0.0288,0.1345,0.1057,resilience_flexibility_agility,3.0,5.0,False
35,50,0.0288,0.1345,0.1057,resilience_flexibility_agility,3.0,5.0,False
4,7,0.2930,0.3785,0.0856,talent_management,3.0,5.0,False


In [ ]:
print(
    "Maximum single-skill gain summary:"
)

display(
    best_single_df[
        "maximum_gain"
    ].describe()
)

print(
    "\nRespondents where at least one "
    "skill increase improved probability:"
)

print(
    (
        best_single_df[
            "maximum_gain"
        ] > 0
    ).sum(),
    "/",
    len(best_single_df)
)

print(
    "\nRespondents crossing threshold:"
)

print(
    best_single_df[
        "crosses_threshold"
    ].sum(),
    "/",
    len(best_single_df)
)

Maximum single-skill gain summary:


,maximum_gain
count,106.000000
mean,0.048735
std,0.027258
min,0.000000
25%,0.027561
50%,0.046560
75%,0.064801
max,0.114483



Respondents where at least one skill increase improved probability:
105 / 106

Respondents crossing threshold:
1 / 106


In [ ]:
multi_cf_rows = []

for count, respondent_index in enumerate(
    below_threshold_df.index,
    start=1
):

    original_row = (
        X.loc[[respondent_index]]
        .copy()
    )

    original_probability = (
        predict_employment_probability(
            original_row
        )
    )

    solutions = (
        search_minimal_skill_counterfactuals(
            original_row,
            max_changed_skills=3
        )
    )

    if len(solutions) > 0:

        best = solutions[0]

        multi_cf_rows.append({
            "respondent_index":
                respondent_index,

            "original_probability":
                original_probability,

            "has_counterfactual":
                True,

            "number_of_skills_changed":
                best[
                    "number_of_skills_changed"
                ],

            "total_skill_increase":
                best[
                    "total_skill_increase"
                ],

            "counterfactual_probability":
                best[
                    "counterfactual_probability"
                ],

            "probability_gain":
                best[
                    "probability_gain"
                ],

            "changed_skills":
                best[
                    "changed_skills"
                ]
        })

    else:

        multi_cf_rows.append({
            "respondent_index":
                respondent_index,

            "original_probability":
                original_probability,

            "has_counterfactual":
                False,

            "number_of_skills_changed":
                None,

            "total_skill_increase":
                None,

            "counterfactual_probability":
                None,

            "probability_gain":
                None,

            "changed_skills":
                None
        })

    if count % 10 == 0:
        print(
            f"Processed {count} / "
            f"{len(below_threshold_df)}"
        )


multi_cf_df = pd.DataFrame(
    multi_cf_rows
)

print("\nCompleted.")

display(
    multi_cf_df.head()
)

Processed 10 / 106


KeyboardInterrupt: 

In [ ]:
n_total = len(
    multi_cf_df
)

n_success = int(
    multi_cf_df[
        "has_counterfactual"
    ].sum()
)

n_failure = (
    n_total - n_success
)

coverage = (
    n_success
    / n_total
    * 100
)

print(
    "Below-threshold respondents:",
    n_total
)

print(
    "Successful counterfactuals:",
    n_success
)

print(
    "No counterfactual within ≤3 skills:",
    n_failure
)

print(
    "Counterfactual coverage:",
    round(coverage, 2),
    "%"
)

In [ ]:
successful_multi_cf = (
    multi_cf_df[
        multi_cf_df[
            "has_counterfactual"
        ]
    ]
    .copy()
)

if len(successful_multi_cf) > 0:

    skills_required = (
        successful_multi_cf[
            "number_of_skills_changed"
        ]
        .value_counts()
        .sort_index()
        .rename_axis(
            "number_of_skills"
        )
        .reset_index(
            name="respondents"
        )
    )

    skills_required[
        "percentage_of_successful_cases"
    ] = (
        skills_required[
            "respondents"
        ]
        / len(successful_multi_cf)
        * 100
    )

    display(
        skills_required.round(2)
    )

else:

    print(
        "No successful counterfactuals "
        "within 3 skill changes."
    )

In [ ]:
if len(successful_multi_cf) > 0:

    print(
        "Total skill-point increase "
        "among successful CFs:"
    )

    display(
        successful_multi_cf[
            "total_skill_increase"
        ].describe()
    )

    print(
        "\nProbability gain:"
    )

    display(
        successful_multi_cf[
            "probability_gain"
        ].describe()
    )